# 🧩 Mini-Lab: Latency and Cost

**Module: Is It Any Good? Quality, Latency, Cost** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** what p50 and p95 latency mean and why averages alone are misleading
2. **Compute** percentile latency from structured log data
3. **Estimate** the cost of running your AI system per 1,000 records using token counts and pricing

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Percentile Latency | The mean hides the slow calls and timeouts, so use percentiles like p50 and p95 to describe typical latency and tail slowness users actually feel. |
| Cost-Per-Unit Estimate | Compute cost per 1,000 records from your own logged counts, then treat price as a variable so you can compare models and prompts fairly. |
| Structured Logging | Logs as records with a run id and consistent per-record fields, so you can trace behavior and search reliably, while redacting sensitive keys and user text by default. |


## Why Averages Lie

Suppose your AI system's average latency is **320 ms**. Sounds fast! But if 5% of requests take **2+ seconds**, those users have a terrible experience — and averages hide that completely.

**Percentile latency** fixes this:
- **p50** (median) — half of requests are faster than this. This is the "typical" experience.
- **p95** — 95% of requests are faster than this. This is the "worst reasonable" experience.

If p50 is 250 ms but p95 is 2,100 ms, you have a tail-latency problem that the average would never reveal.

Similarly, **cost** is easy to ignore until you scale. Knowing the cost per 1,000 records lets you forecast what happens when you go from 100 test runs to 100,000 production requests.

## Step 1 — Simulate Structured Logs

In production, every API call should log its latency and token usage in a structured (machine-readable) format. Here we simulate 40 logged API calls — each is a dictionary with the fields you'd capture in a real system.

In [1]:
import random
random.seed(42)

def simulate_logs(n=40):
    """Simulate structured logs from an LLM-based classification system."""
    logs = []
    for i in range(n):
        # Most calls are fast; a few are slow (simulating tail latency)
        if random.random() < 0.90:
            latency_ms = random.uniform(180, 450)   # typical range
        else:
            latency_ms = random.uniform(1200, 3500)  # slow outliers

        prompt_tokens = random.randint(120, 280)
        completion_tokens = random.randint(5, 30)

        logs.append({
            "request_id": i + 1,
            "latency_ms": round(latency_ms, 1),
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "model": "gpt-4o-mini",
            "status": "success",
        })
    return logs

logs = simulate_logs()

# Show a few example log entries
print("Example structured log entries:")
for entry in logs[:3]:
    print(entry)

Example structured log entries:
{'request_id': 1, 'latency_ms': 186.8, 'prompt_tokens': 190, 'completion_tokens': 12, 'model': 'gpt-4o-mini', 'status': 'success'}
{'request_id': 2, 'latency_ms': 378.8, 'prompt_tokens': 259, 'completion_tokens': 7, 'model': 'gpt-4o-mini', 'status': 'success'}
{'request_id': 3, 'latency_ms': 188.6, 'prompt_tokens': 143, 'completion_tokens': 11, 'model': 'gpt-4o-mini', 'status': 'success'}


Each log entry captures exactly what you need for evaluation:
- **`latency_ms`** — how long the API call took
- **`prompt_tokens`** / **`completion_tokens`** — what it cost in tokens
- **`model`** — which model was used (pricing depends on this)
- **`status`** — whether the call succeeded

This is **structured logging** — machine-readable records you can analyze programmatically, not scattered `print()` statements.

## Step 2 — Compute p50 and p95 Latency

> 🧩 **p50 / p95 latency.** The percentiles from the video, now computed from your logs.

We sort all latency values and pick the values at the 50th and 95th percentile positions.

In [2]:
def percentile(values, p):
    """Compute the p-th percentile of a list of numbers."""
    sorted_vals = sorted(values)
    index = int(len(sorted_vals) * p / 100)
    index = min(index, len(sorted_vals) - 1)  # clamp to last element
    return sorted_vals[index]

latencies = [entry["latency_ms"] for entry in logs]

p50 = percentile(latencies, 50)
p95 = percentile(latencies, 95)
avg = sum(latencies) / len(latencies)

print(f"Requests analyzed: {len(latencies)}")
print(f"Average latency:   {avg:.0f} ms")
print(f"p50 latency:       {p50:.0f} ms   ← the typical user experience")
print(f"p95 latency:       {p95:.0f} ms   ← the worst 5% experience")
print()
if p95 > 3 * p50:
    print("⚠️  Large gap between p50 and p95 — you have a tail-latency problem.")
else:
    print("✅  p95 is within a reasonable range of p50.")

Requests analyzed: 40
Average latency:   442 ms
p50 latency:       329 ms   ← the typical user experience
p95 latency:       2690 ms   ← the worst 5% experience

⚠️  Large gap between p50 and p95 — you have a tail-latency problem.


Notice how the **average** sits somewhere between p50 and p95 — it doesn't tell you about the slow outliers that p95 exposes. Always report both percentiles, not just the mean.

## Step 3 — Estimate Cost per 1,000 Records

> 🧩 **Cost per 1,000 records.** Compute cost per 1,000 records from your own logged counts, then treat price as a variable.
> Watch the next output for cost per 1,000 records, not a single-request price.

To estimate cost, we need two things:
1. **Average tokens per request** (from our logs)
2. **Model pricing** (from the provider's pricing page)

We'll use the pricing for `gpt-4o-mini` as an example.


In [3]:
# Model pricing (per 1M tokens) — check provider's pricing page for current rates
PRICING = {
    "gpt-4o-mini": {
        "input_per_1M":  0.15,   # $ per 1M input tokens
        "output_per_1M": 0.60,   # $ per 1M output tokens
    },
}

model = logs[0]["model"]
price = PRICING[model]

# Calculate average tokens per request from our logs
avg_prompt_tokens     = sum(e["prompt_tokens"] for e in logs) / len(logs)
avg_completion_tokens = sum(e["completion_tokens"] for e in logs) / len(logs)

# Cost per single request
cost_per_request = (
    avg_prompt_tokens     * price["input_per_1M"]  / 1_000_000
    + avg_completion_tokens * price["output_per_1M"] / 1_000_000
)

# Scale to 1,000 records
cost_per_1000 = cost_per_request * 1000

print(f"Model: {model}")
print(f"Avg prompt tokens:     {avg_prompt_tokens:.0f}")
print(f"Avg completion tokens: {avg_completion_tokens:.0f}")
print(f"Cost per request:      ${cost_per_request:.6f}")
print(f"Cost per 1,000:        ${cost_per_1000:.4f}")
print(f"Cost per 100,000:      ${cost_per_request * 100_000:.2f}")

Model: gpt-4o-mini
Avg prompt tokens:     205
Avg completion tokens: 19
Cost per request:      $0.000042
Cost per 1,000:        $0.0420
Cost per 100,000:      $4.20


## Step 4 — Generate the Latency & Cost Report

Let's put everything into a single report — the kind of summary you'd include in a ship decision or hand to a stakeholder.

In [4]:
from IPython.display import display, Markdown
def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

report = f"""### 📋 Latency & Cost Report

| Metric | Value |
|--------|-------|
| Requests analyzed | {len(logs)} |
| Model | `{model}` |
| **p50 latency** | **{p50:.0f} ms** |
| **p95 latency** | **{p95:.0f} ms** |
| Average latency | {avg:.0f} ms |
| Avg prompt tokens | {avg_prompt_tokens:.0f} |
| Avg completion tokens | {avg_completion_tokens:.0f} |
| **Cost per 1,000 records** | **${cost_per_1000:.4f}** |
| Projected cost per 100K | ${cost_per_request * 100_000:.2f} |

#### Interpretation

- **Typical experience (p50):** {p50:.0f} ms — most users see fast responses.
- **Tail latency (p95):** {p95:.0f} ms — 1 in 20 requests takes this long or more.
- **Cost at scale:** Processing 100K records would cost ~${cost_per_request * 100_000:.2f} with `{model}`.

> 💡 **Tip:** If p95 latency is too high, consider request timeouts and retries.
> If cost is too high, consider a cheaper model or reducing prompt length.
"""

md(report)

### 📋 Latency & Cost Report

| Metric | Value |
|--------|-------|
| Requests analyzed | 40 |
| Model | `gpt-4o-mini` |
| **p50 latency** | **329 ms** |
| **p95 latency** | **2690 ms** |
| Average latency | 442 ms |
| Avg prompt tokens | 205 |
| Avg completion tokens | 19 |
| **Cost per 1,000 records** | **$0.0420** |
| Projected cost per 100K | $4.20 |

#### Interpretation

- **Typical experience (p50):** 329 ms — most users see fast responses.
- **Tail latency (p95):** 2690 ms — 1 in 20 requests takes this long or more.
- **Cost at scale:** Processing 100K records would cost ~$4.20 with `gpt-4o-mini`.

> 💡 **Tip:** If p95 latency is too high, consider request timeouts and retries.
> If cost is too high, consider a cheaper model or reducing prompt length.


## 🎯 Summary

### Key Takeaways

1. **Percentile latency (p50/p95)** — report median and 95th-percentile latency instead of averages; averages hide slow outliers that hurt real users
2. **Cost-per-unit estimate** — multiply average token usage by model pricing to forecast cost at scale; a cheap-per-request model can still be expensive at 100K records
3. **Structured logging** — log every API call's latency, token counts, model, and status in a machine-readable format so you can compute these metrics after the fact